In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
import pandas as pd
from PIL import Image
import numpy as np
import os
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler
import random
import datetime
import pickle
import warnings
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

class PROIConfig:
    def __init__(self):
        
        self.use_val_set = False # Whether to load the validation set
        
        # Data loading mode configuration: 'pkl' for serialized data files, 'xlsx' for original spreadsheets.
        self.data_mode = 'pkl'
        self.train_pkl_path = '../autodl-tmp/data/tw_train_3img.pkl'
        #self.val_pkl_path = 'autodl-tmp/data/tw_val_3img.pkl' 
        self.test_pkl_path = '../autodl-tmp/data/tw_test_3img.pkl'


        # self.data_mode = 'xlsx'
        # self.train_excel_path = 'tw15_train.xlsx'
        # self.train_img_path = 'tw15_image_224'
        # self.val_excel_path = 'tw15_val.xlsx' 
        # self.val_img_path = 'tw15_image_224'
        # self.test_excel_path = 'tw15_test.xlsx'
        # self.test_img_path = 'tw15_image_224'
        
        self.log_path = 'tw_PROI_Log.txt'
        self.best_model_path = '../autodl-tmp/best_model/PROI_tw.pth'
        os.makedirs(os.path.dirname(self.best_model_path), exist_ok=True)

        self.clip_model_name = '../model/clip-vit-base-patch16'
        self.text_max_length = 77
        self.image_size = 224
        self.max_img_num = 1
        
        self.RANDOM_SEED = 42
        self.batch_size = 32
        self.num_epochs = 30
        
        self.learning_rate = 3.5e-05
        self.weight_decay = 0.036

        self.gamma_scale = 10.0
        self.alpha_text = 0.5
        self.beta_smooth_t = 0.8
        self.beta_smooth_i = 0.8
        self.lambda_1_cls = 1.0
        self.lambda_2_contrastive = 0.4
        self.lambda_3_saturation = 0.4
        self.temperature = 0.07

        self.warmup_epochs = 0
        self.patience = 10

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def conditional_orthogonal_projection(u, v):
    v_norm_sq = torch.sum(v * v, dim=-1, keepdim=True)
    v_norm_sq = torch.where(v_norm_sq == 0, torch.ones_like(v_norm_sq), v_norm_sq)
    dot_product = torch.sum(u * v, dim=-1, keepdim=True)
    proj_scalar = dot_product / v_norm_sq
    proj_vector = proj_scalar * v
    condition = dot_product > 0
    return torch.where(condition, proj_vector, torch.zeros_like(proj_vector))

def absolute_proj_length(u, v):
    dot_product = torch.sum(u * v, dim=-1, keepdim=True)
    v_norm = torch.norm(v, p=2, dim=-1, keepdim=True)
    v_norm = torch.where(v_norm < 1e-9, torch.ones_like(v_norm), v_norm)
    scalar_proj = dot_product / v_norm
    return torch.abs(scalar_proj).squeeze(-1)

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07, contrast_mode='all', base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        device = features.device
        if len(features.shape) < 3: raise ValueError('`features` needs to be [bsz, n_views, ...]')
        if len(features.shape) > 3: features = features.view(features.shape[0], features.shape[1], -1)
        batch_size = features.shape[0]
        if labels is not None and mask is not None: raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None: mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size: raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else: mask = mask.float().to(device)
        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one': 
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all': 
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else: raise ValueError('Unknown mode: {}'.format(self.contrast_mode))
        anchor_dot_contrast = torch.div(torch.matmul(anchor_feature, contrast_feature.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()
        mask = mask.repeat(anchor_count, contrast_count)
        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size * anchor_count).view(-1, 1).to(device), 0)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()
        return loss

class NewsDataset(Dataset):
    def __init__(self, config, excel_path, img_folder, is_train=True):
        self.config = config
        self.df = pd.read_excel(excel_path)
        self.img_folder = img_folder
        self.is_train = is_train
        self.news_cols = ['news_g1', 'news_g2', 'news_m', 'news_q']
        if not self.is_train:
            for col in self.news_cols:
                if col not in self.df.columns: self.df[col] = ""
                 
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text']) if pd.notna(row['text']) else ""
        news_texts = [str(row[col]) if pd.notna(row[col]) else "" for col in self.news_cols]
        img_names = str(row['img_name']).split(',') if pd.notna(row['img_name']) else []
        images, valid_img_count = [], 0
        for i in range(self.config.max_img_num):
            if i < len(img_names) and img_names[i]:
                try:
                    images.append(Image.open(os.path.join(self.img_folder, img_names[i])).convert("RGB"))
                    valid_img_count += 1
                except FileNotFoundError: 
                    images.append(Image.new('RGB', (self.config.image_size, self.config.image_size)))
            else: 
                images.append(Image.new('RGB', (self.config.image_size, self.config.image_size)))
        i_mask = torch.zeros(self.config.max_img_num)
        i_mask[:valid_img_count] = 1
        label = int(row['label']) if 'label' in self.df.columns else -1
        return {'text': text, 'news_texts': news_texts, 'images': images, 'i_mask': i_mask, 'label': torch.tensor(label, dtype=torch.long)}

class PKLDataset(Dataset):
    def __init__(self, config, pkl_path, is_train=True):
        self.config = config
        self.is_train = is_train
        print(f"Loading dataset: {pkl_path} ...")
        with open(pkl_path, 'rb') as f:
            self.data = pickle.load(f)
            
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        news_texts = item['news_texts']
        label = item['label']
        raw_images = item['images']
        target_num = self.config.max_img_num
        current_images = raw_images[:target_num]
        final_images = []
        valid_img_count = 0
        for img in current_images:
            final_images.append(img)
            valid_img_count += 1
        pad_num = target_num - valid_img_count
        for _ in range(pad_num):
            final_images.append(Image.new('RGB', (self.config.image_size, self.config.image_size)))
        i_mask = torch.zeros(target_num)
        i_mask[:valid_img_count] = 1
        return {
            'text': text, 
            'news_texts': news_texts, 
            'images': final_images, 
            'i_mask': i_mask, 
            'label': torch.tensor(label, dtype=torch.long)
        }

class MultiModalEmbedder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.clip_model = CLIPModel.from_pretrained(config.clip_model_name)
        self.processor = CLIPProcessor.from_pretrained(config.clip_model_name)

    def forward(self, text, news_texts, images):
        device = self.config.device
        text_inputs = self.processor(text=text, return_tensors="pt", padding=True, truncation=True, max_length=self.config.text_max_length).to(device)
        T_cls = self.clip_model.get_text_features(**text_inputs)
        flat_news_texts = [n for sublist in news_texts for n in sublist]
        news_inputs = self.processor(text=flat_news_texts, return_tensors="pt", padding=True, truncation=True, max_length=self.config.text_max_length).to(device)
        N_cls_flat = self.clip_model.get_text_features(**news_inputs)
        N_cls = N_cls_flat.view(-1, 4, T_cls.shape[-1])
        flat_images = [img for sublist in images for img in sublist]
        image_inputs = self.processor(images=flat_images, return_tensors="pt", padding=True).to(device)
        I_cls = self.clip_model.get_image_features(**image_inputs).view(-1, self.config.max_img_num, T_cls.shape[-1])
        return T_cls, N_cls, I_cls

class DisentanglementAndDistillationModule(nn.Module):
    def __init__(self, config, input_dim=512):
        super().__init__()
        self.config = config
        self.input_dim = input_dim
        self.sup_con_loss = SupConLoss(temperature=config.temperature)
        self.cos_sim = nn.CosineSimilarity(dim=-1)
        self.scale_factor = config.gamma_scale
        self.alpha = config.alpha_text
        self.event_direction_extractor = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())

    def forward(self, T_cls, N_cls, I_cls, i_mask):
        device = self.config.device
        T_unit = F.normalize(T_cls, p=2, dim=-1)
        I_unit = F.normalize(I_cls, p=2, dim=-1)
        T_norm_scaled = T_unit * self.scale_factor
        I_norm_scaled = I_unit * self.scale_factor
        signal_text = self.event_direction_extractor(T_norm_scaled)
        interaction_scaled = (1 - self.alpha) * I_norm_scaled + self.alpha * T_norm_scaled.unsqueeze(1)
        signal_image = self.event_direction_extractor(interaction_scaled)
        student_te = T_norm_scaled * signal_text 
        i_mask_expanded = i_mask.unsqueeze(-1)
        student_ie = I_norm_scaled * signal_image * i_mask_expanded
        summed_student_ie = torch.sum(student_ie, dim=1)
        teacher_n = F.normalize(N_cls, p=2, dim=-1)
        student_n_raw = student_te + summed_student_ie
        student_n = F.normalize(student_n_raw, p=2, dim=-1)
        features_joint = torch.cat([student_n.unsqueeze(1), teacher_n], dim=1)
        loss_contrastive = self.sup_con_loss(features_joint)
        loss_saturation = signal_text.mean() + signal_image.mean()
        l1 = absolute_proj_length(student_te, student_n_raw)
        l2 = absolute_proj_length(summed_student_ie, student_n_raw)
        c = l1 / (l1 + l2 + 1e-9)
        raw_IE = conditional_orthogonal_projection(I_norm_scaled, student_ie)
        raw_IB = I_norm_scaled - raw_IE
        IE = raw_IE * i_mask_expanded 
        IB = raw_IB * i_mask_expanded 
        TE = conditional_orthogonal_projection(T_norm_scaled, student_te) 
        TS = T_norm_scaled - TE 
        loss_completeness = torch.tensor(0.0, device=device) 
        return TS, TE, loss_contrastive, loss_completeness, IE, IB, I_cls, c, loss_saturation, T_norm_scaled, I_norm_scaled

class FeatureInteractionModule(nn.Module):
    def __init__(self, config, input_dim=512, mlp_output_dim=512):
        super().__init__()
        
    def forward(self, TE, TS, IE_pool, IB_pool):
        return TE * IB_pool, TS * IE_pool, TE * TS, IE_pool * IB_pool

class GeometryBasedDynamicWeighting(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.smooth_factor_t = config.beta_smooth_t
        self.smooth_factor_i = config.beta_smooth_i

    def forward(self, T_cls, TS, TE, IE, IB, I_cls, i_mask, c):
        sim_ts_tcls = F.cosine_similarity(TS, T_cls, dim=-1)
        sim_te_tcls = F.cosine_similarity(TE, T_cls, dim=-1)
        raw_k1 = sim_ts_tcls / (sim_ts_tcls + sim_te_tcls + 1e-9)
        K1 = self.smooth_factor_t * raw_k1 + (1 - self.smooth_factor_t) * 0.5
        sim_ib_icls = F.cosine_similarity(IB, I_cls, dim=-1) 
        sim_ie_icls = F.cosine_similarity(IE, I_cls, dim=-1) 
        raw_K2_per_image = sim_ib_icls / (sim_ib_icls + sim_ie_icls + 1e-9)
        smoothed_K2_per_image = self.smooth_factor_i * raw_K2_per_image + (1 - self.smooth_factor_i) * 0.5
        norm_ie = torch.norm(IE, p=2, dim=-1) 
        img_weights = norm_ie * i_mask 
        weights_sum = img_weights.sum(dim=1, keepdim=True) + 1e-9
        normalized_weights = img_weights / weights_sum 
        weights_expanded = normalized_weights.unsqueeze(-1)
        IE_pool = (IE * weights_expanded).sum(dim=1)
        IB_pool = (IB * weights_expanded).sum(dim=1)
        K2 = (smoothed_K2_per_image * normalized_weights).sum(dim=1)
        k31 = c / (1 - K1 + 1e-9)
        k32 = (1 - c) / (1 - K2 + 1e-9)
        raw_K3 = k31 / (k31 + k32 + 1e-9)
        K3 = self.smooth_factor_t * raw_K3 + (1 - self.smooth_factor_t) * 0.5
        return K1, K2, K3, IE_pool, IB_pool

class FeatureAggregationAndOutput(nn.Module):
    def __init__(self, config, input_dim_z1=512, input_dim_g=512, num_classes=2):
        super().__init__()
        self.config = config
        self.classifier = nn.Sequential(nn.Linear(512, 2))
        self.eps = 1e-9

    def forward(self, TE, TS, IE_pool, IB_pool, K1, K2, K3, Gx_tuple):
        K1, K2, K3 = K1.unsqueeze(1), K2.unsqueeze(1), K3.unsqueeze(1)
        A = K3 * (1 - K1)
        B = (1 - K3) * (1 - K2)
        K4 = A / (A + B + self.eps)
        one_minus_K4 = 1.0 - K4
        TE_IB, TS_IE, TE_TS, IE_IB = Gx_tuple
        Z2_raw = one_minus_K4 * TS_IE + K4 * TE_TS  
        Z3_raw = K4 * TE_IB + one_minus_K4 * IE_IB  
        Z1 = K3 * (1 - K1) * TE + (1 - K3) * (1 - K2) * IE_pool
        Z = Z1 + K3 * K1 * Z2_raw + (1 - K3) * K2 * Z3_raw
        return self.classifier(Z)

class PROI(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedder = MultiModalEmbedder(config)
        self.disentangler = DisentanglementAndDistillationModule(config)
        self.interactor = FeatureInteractionModule(config)
        self.weigher = GeometryBasedDynamicWeighting(config)
        self.aggregator = FeatureAggregationAndOutput(config)

    def forward(self, batch, is_train=True):
        i_mask = batch['i_mask'].to(self.config.device)
        has_news = not (not is_train and all(all(s == "" for s in nt) for nt in batch['news_texts']))
        if has_news: 
            T_cls, N_cls, I_cls = self.embedder(batch['text'], batch['news_texts'], batch['images'])
        else:
            T_cls, _, I_cls = self.embedder(batch['text'], [["", "", "", ""]] * len(batch['text']), batch['images'])
            N_cls = torch.zeros(T_cls.shape[0], 4, T_cls.shape[1]).to(self.config.device)
        TS, TE, loss_contrastive, loss_completeness, IE, IB, I_cls, c, loss_saturation, T_norm_scaled, I_norm_scaled = self.disentangler(T_cls, N_cls, I_cls, i_mask)
        if not has_news: loss_contrastive = torch.tensor(0.0).to(self.config.device)
        K1, K2, K3, IE_pool, IB_pool = self.weigher(T_norm_scaled, TS, TE, IE, IB, I_norm_scaled, i_mask, c)
        Gx_tuple = self.interactor(TE, TS, IE_pool, IB_pool)
        logits = self.aggregator(TE, TS, IE_pool, IB_pool, K1, K2, K3, Gx_tuple)
        return logits, loss_contrastive, loss_completeness, loss_saturation

def run_epoch(model, dataloader, optimizer, criterion, scaler, config, epoch_num, is_train=True, prefix="Val"):
    model.train() if is_train else model.eval()
    total_loss, total_cls_loss, total_contrastive_loss, correct_predictions, total_samples = 0.0, 0.0, 0.0, 0, 0
    total_loss_completeness, total_saturation_loss = 0.0, 0.0
    all_labels, all_predictions = [], []
    phase = "Training" if is_train else f"{prefix}"
    desc_str = f"Epoch {epoch_num+1}/{config.num_epochs}" if epoch_num is not None else "Testing"
    progress_bar = tqdm(dataloader, desc=f"{desc_str} - {phase}", leave=False)
    for batch in progress_bar:
        labels = batch['label'].to(config.device)
        with autocast(enabled=is_train):
            logits, loss_contrastive, loss_completeness, loss_saturation = model(batch, is_train=is_train)
            loss_cls = criterion(logits, labels)
            loss = config.lambda_1_cls * loss_cls + config.lambda_2_contrastive * loss_contrastive + config.lambda_3_saturation * loss_saturation
        if is_train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * labels.size(0)
        total_cls_loss += loss_cls.item() * labels.size(0)
        total_contrastive_loss += loss_contrastive.item() * labels.size(0)
        total_loss_completeness += loss_completeness.item() * labels.size(0)
        total_saturation_loss += loss_saturation.item() * labels.size(0)
        _, predicted = torch.max(logits, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_samples += labels.size(0)
        all_labels.append(labels.cpu())
        all_predictions.append(predicted.cpu())
        progress_bar.set_postfix(loss=total_loss / total_samples, acc=correct_predictions / total_samples)
    avg_loss = total_loss / total_samples
    avg_cls_loss = total_cls_loss / total_samples
    avg_contrastive_loss = total_contrastive_loss / total_samples
    avg_saturation_loss = total_saturation_loss / total_samples
    accuracy = correct_predictions / total_samples
    all_labels = torch.cat(all_labels).numpy()
    all_predictions = torch.cat(all_predictions).numpy()
    precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    report = classification_report(all_labels, all_predictions, output_dict=True, zero_division=0, labels=np.unique(all_predictions))
    neg_metrics = report.get('0', {'precision': 0, 'recall': 0, 'f1-score': 0})
    pos_metrics = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
    neg_mask, pos_mask = (all_labels == 0), (all_labels == 1)
    neg_acc = (all_predictions[neg_mask] == all_labels[neg_mask]).sum() / neg_mask.sum() if neg_mask.sum() > 0 else 0
    pos_acc = (all_predictions[pos_mask] == all_labels[pos_mask]).sum() / pos_mask.sum() if pos_mask.sum() > 0 else 0
    metrics = {
        "avg_loss": avg_loss, "avg_cls_loss": avg_cls_loss, "avg_contrastive_loss": avg_contrastive_loss, "avg_saturation_loss": avg_saturation_loss,
        "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
        "neg_acc": neg_acc, "neg_p": neg_metrics['precision'], "neg_r": neg_metrics['recall'], "neg_f1": neg_metrics['f1-score'],
        "pos_acc": pos_acc, "pos_p": pos_metrics['precision'], "pos_r": pos_metrics['recall'], "pos_f1": pos_metrics['f1-score'],
    }
    return metrics

def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def collate_fn(batch):
    return {
        'text': [item['text'] for item in batch], 
        'news_texts': [item['news_texts'] for item in batch], 
        'images': [item['images'] for item in batch], 
        'i_mask': torch.stack([item['i_mask'] for item in batch]), 
        'label': torch.stack([item['label'] for item in batch])
    }

def main():
    config = PROIConfig()
    
    with open(config.log_path, 'w', encoding='utf-8') as log_file:
        current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_file.write(f"{'='*60}\nStart Time: {current_time}\nRandom Seed: {config.RANDOM_SEED}\n{'='*60}\n\n")

    set_seed(config.RANDOM_SEED)

    if config.data_mode == 'pkl':
        train_dataset = PKLDataset(config, config.train_pkl_path, is_train=True)
        labels = [item['label'] for item in train_dataset.data]
        class_counts = np.bincount(labels)
        class_weights = 1. / class_counts
        sample_weights = np.array([class_weights[label] for label in labels])
        sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, sampler=sampler, collate_fn=collate_fn, pin_memory=True)
        
        test_dataset = PKLDataset(config, config.test_pkl_path, is_train=False)
        test_seed = 2025
        test_gen = torch.Generator(device='cpu')
        test_gen.manual_seed(test_seed)
        test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn, generator=test_gen)
        
        if config.use_val_set:
            val_dataset = PKLDataset(config, config.val_pkl_path, is_train=False)
            val_seed = 666
            val_gen = torch.Generator(device='cpu')
            val_gen.manual_seed(val_seed)
            val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn, generator=val_gen)
    else:
        train_dataset = NewsDataset(config, config.train_excel_path, config.train_img_path, is_train=True)
        labels = train_dataset.df['label'].values
        class_counts = np.bincount(labels)
        class_weights = 1. / class_counts
        sample_weights = np.array([class_weights[label] for label in labels])
        sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, sampler=sampler, collate_fn=collate_fn, pin_memory=True)
        
        test_dataset = NewsDataset(config, config.test_excel_path, config.test_img_path, is_train=False)
        test_seed = 2025
        test_gen = torch.Generator(device='cpu')
        test_gen.manual_seed(test_seed)
        test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn, generator=test_gen)

        if config.use_val_set:
            val_dataset = NewsDataset(config, config.val_excel_path, config.val_img_path, is_train=False)
            val_seed = 666
            val_gen = torch.Generator(device='cpu')
            val_gen.manual_seed(val_seed)
            val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn, generator=val_gen)

    model = PROI(config).to(config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()

    best_eval_f1 = 0.0
    patience_counter = 0
    
    print(f"\n--- Starting Training (Total Epochs: {config.num_epochs}) ---")
    
    for epoch in range(config.num_epochs):
        train_metrics = run_epoch(model, train_loader, optimizer, criterion, scaler, config, epoch, is_train=True)
        
        with torch.no_grad():
            if config.use_val_set:
                eval_metrics = run_epoch(model, val_loader, None, criterion, scaler, config, epoch, is_train=False, prefix="Validation")
                eval_name = "Validation"
            else:
                print("\nValidation set not detected. The current test results are used for early stopping.")
                eval_metrics = run_epoch(model, test_loader, None, criterion, scaler, config, epoch, is_train=False, prefix="Test")
                eval_name = "Test"
                
        with open(config.log_path, 'a', encoding='utf-8') as log_file:
            train_log = f"Epoch {epoch+1} Train Results -> Total Loss: {train_metrics['avg_loss']:.4f} | Cls Loss: {train_metrics['avg_cls_loss']:.4f} | Contrastive Loss: {train_metrics['avg_contrastive_loss']:.4f} | Saturation Loss: {train_metrics['avg_saturation_loss']:.4f}\n"
            train_log += f"  Overall -> ACC: {train_metrics['accuracy']:.4f} | P: {train_metrics['precision']:.4f} | R: {train_metrics['recall']:.4f} | F1: {train_metrics['f1']:.4f}\n"
            train_log += f"  Positive Samples -> ACC: {train_metrics['pos_acc']:.4f} | P: {train_metrics['pos_p']:.4f} | R: {train_metrics['pos_r']:.4f} | F1: {train_metrics['pos_f1']:.4f}\n"
            train_log += f"  Negative Samples -> ACC: {train_metrics['neg_acc']:.4f} | P: {train_metrics['neg_p']:.4f} | R: {train_metrics['neg_r']:.4f} | F1: {train_metrics['neg_f1']:.4f}\n"

            eval_log = f"Epoch {epoch+1} {eval_name} Results -> Total Loss: {eval_metrics['avg_loss']:.4f} | Cls Loss: {eval_metrics['avg_cls_loss']:.4f} | Contrastive Loss: {eval_metrics['avg_contrastive_loss']:.4f} | Saturation Loss: {eval_metrics['avg_saturation_loss']:.4f}\n"
            eval_log += f"  Overall -> ACC: {eval_metrics['accuracy']:.4f} | P: {eval_metrics['precision']:.4f} | R: {eval_metrics['recall']:.4f} | F1: {eval_metrics['f1']:.4f}\n"
            eval_log += f"  Positive Samples -> ACC: {eval_metrics['pos_acc']:.4f} | P: {eval_metrics['pos_p']:.4f} | R: {eval_metrics['pos_r']:.4f} | F1: {eval_metrics['pos_f1']:.4f}\n"
            eval_log += f"  Negative Samples -> ACC: {eval_metrics['neg_acc']:.4f} | P: {eval_metrics['neg_p']:.4f} | R: {eval_metrics['neg_r']:.4f} | F1: {eval_metrics['neg_f1']:.4f}\n"
            
            log_file.write(f"\n[Epoch {epoch+1}/{config.num_epochs}]\n")
            log_file.write("---------- Train Results ----------\n")
            log_file.write(train_log)
            log_file.write(f"\n---------- {eval_name} Results ----------\n")
            log_file.write(eval_log)
            log_file.write("\n" + "-" * 80 + "\n")

        print(f"Epoch {epoch+1}/{config.num_epochs} | Train F1: {train_metrics['f1']:.4f} | {eval_name} F1: {eval_metrics['f1']:.4f}")
        current_f1 = eval_metrics['f1']
        
        if epoch >= config.warmup_epochs:
            if current_f1 > best_eval_f1:
                best_eval_f1 = current_f1
                patience_counter = 0
                torch.save(model.state_dict(), config.best_model_path)
                print(f"  [New Best] Model saved to {config.best_model_path} ({eval_name} F1: {best_eval_f1:.4f})")
                with open(config.log_path, 'a', encoding='utf-8') as log_file:
                    log_file.write(f"*** New Best Model Saved (Epoch {epoch+1}, {eval_name} F1: {best_eval_f1:.4f}) ***\n")
            else:
                patience_counter += 1
                print(f"  [Patience] {patience_counter}/{config.patience}")
                if patience_counter >= config.patience:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs.")
                    with open(config.log_path, 'a', encoding='utf-8') as log_file:
                        log_file.write(f"\nEarly stopping triggered. Best {eval_name} F1: {best_eval_f1:.4f}\n")
                    break

    print(f"\n--- Loading Best Model for Final Testing ---")
    model.load_state_dict(torch.load(config.best_model_path))
    
    with torch.no_grad():
        final_test_metrics = run_epoch(model, test_loader, None, criterion, scaler, config, None, is_train=False, prefix="Test")

    with open(config.log_path, 'a', encoding='utf-8') as log_file:
        test_log = f"Final Test Results (Best Model) -> Total Loss: {final_test_metrics['avg_loss']:.4f}\n"
        test_log += f"  Overall -> ACC: {final_test_metrics['accuracy']:.4f} | P: {final_test_metrics['precision']:.4f} | R: {final_test_metrics['recall']:.4f} | F1: {final_test_metrics['f1']:.4f}\n"
        test_log += f"  Positive Samples -> ACC: {final_test_metrics['pos_acc']:.4f} | P: {final_test_metrics['pos_p']:.4f} | R: {final_test_metrics['pos_r']:.4f} | F1: {final_test_metrics['pos_f1']:.4f}\n"
        test_log += f"  Negative Samples -> ACC: {final_test_metrics['neg_acc']:.4f} | P: {final_test_metrics['neg_p']:.4f} | R: {final_test_metrics['neg_r']:.4f} | F1: {final_test_metrics['neg_f1']:.4f}\n"
        
        log_file.write("\n" + "="*80 + "\n")
        log_file.write("FINAL TEST RESULTS\n")
        log_file.write(test_log)
        log_file.write("="*80 + "\n")

    print(f"Test Complete. Final Test F1: {final_test_metrics['f1']:.4f}")

if __name__ == '__main__':
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


Loading dataset: ../autodl-tmp/data/tw_train_3img.pkl ...
Loading dataset: ../autodl-tmp/data/tw_test_3img.pkl ...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



--- Starting Training (Total Epochs: 30) ---


Epoch 1/30 - Training:   0%|          | 0/286 [00:00<?, ?it/s]


Validation set not detected. The current test results are used for early stopping.


... (44 outputs left out)

Epoch 12/30 - Test:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 12/30 | Train F1: 0.9998 | Test F1: 0.7564
  [Patience] 10/10

Early stopping triggered after 12 epochs.

--- Loading Best Model for Final Testing ---


Testing - Test:   0%|          | 0/57 [00:00<?, ?it/s]

Test Complete. Final Test F1: 0.9495
